# 4. Honest confidence, and the ternary question

**Calibration:** when a model says "90% sure", is it right about 90% of the time? Answers are sorted into 15
bands by confidence; the gaps between average confidence and accuracy are combined into one number, the
**expected calibration error (ECE)**. Lower is better.

**Temperature scaling** fixes a uniform over- or under-confidence with a single number T. To avoid tuning on the
reported photos, T is fitted on the even-numbered photos and ECE is reported on the odd-numbered ones.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run from the repository root
sys.path.insert(0, "src")
import numpy as np
from IPython.display import Image, display

import ast
def show(path, *names):
    """Print the real source of functions/classes from a file in src/ (read directly, never copied)."""
    src = open(path).read(); lines = src.splitlines(); tree = ast.parse(src)
    nodes = {n.name: n for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.ClassDef))}
    for name in names:
        n = nodes[name]; start = n.decorator_list[0].lineno if n.decorator_list else n.lineno
        print(f"# ---- {path}, lines {start}-{n.end_lineno}")
        print("\n".join(lines[start - 1:n.end_lineno])); print()

def show_lines(path, start_text, end_text):
    """Print a block of a script, from the line containing start_text to the line containing end_text."""
    lines = open(path).read().splitlines()
    i = next(k for k, l in enumerate(lines) if start_text in l)
    j = next(k for k, l in enumerate(lines[i + 1:], i + 1) if end_text in l)
    print(f"# ---- {path}, lines {i + 1}-{j + 1}"); print("\n".join(lines[i:j + 1]))

In [2]:
show("src/rescore.py", "ece", "summarise", "fit_temperature")

# ---- src/rescore.py, lines 147-152
def ece(conf, correct, bins=15):
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return 100.0 * e

# ---- src/rescore.py, lines 155-159
def summarise(logits, labels, T=1.0):
    p = torch.softmax(logits / T, 1); conf, pred = p.max(1)
    conf, pred = conf.cpu().numpy(), pred.cpu().numpy(); ok = pred == labels
    return {"top1": float(100 * ok.mean()), "ece": float(ece(conf, ok.astype(float))),
            "mean_conf": float(100 * conf.mean()), "correct": ok.tolist()}

# ---- src/rescore.py, lines 162-170
def fit_temperature(logits, labels):
    """One scalar T minimising NLL (Guo et al. 2017), LBFGS on log T."""
    y = torch.as_tensor(labels, device=logits.device)
    logT = torch.zeros(1, device=logits.device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=20

In [3]:
T = json.load(open("evidence/rescore/rescore.json"))["temperature"]
print(f"{'model':<14}{'ECE before':>11}{'ECE after':>11}{'T':>8}")
for k in ["deit_s_fp32"] + [f"{r}-s{s}" for r in "ABCD" for s in (0, 1, 2)]:
    t = T[k]; print(f"{k:<14}{t['ece_odd_before']:>11.2f}{t['ece_odd_after']:>11.2f}{t['T']:>8.3f}")

model          ECE before  ECE after       T
deit_s_fp32          8.56       3.40   0.833
A-s0                 4.43       4.08   0.874
A-s1                 4.39       4.08   0.873
A-s2                 4.34       4.11   0.873
B-s0                 5.31       4.03   0.859
B-s1                 5.35       4.00   0.860
B-s2                 5.65       3.74   0.860
C-s0                 4.56       4.00   0.874
C-s1                 4.61       4.32   0.870
C-s2                 4.60       4.22   0.871
D-s0                11.84       3.14   0.802
D-s1                12.00       2.97   0.801
D-s2                12.05       2.92   0.801


**Reading it:** the distilled model (D) is the worst calibrated before correction (about 12) and the best after
(about 3), so its problem is a simple uniform offset that one number removes.

## 4.2 Is ternary specially weak on noise, or just a weaker model?

A model that is better on clean photos is usually better on damaged ones by a predictable amount
(Miller et al., 2021). Seven DeiT-Small models of slightly different quality (route A at five group sizes, plus
routes A and C) draw that trend line. If ternary falls below the line on noise, it is specially weak on noise.

The rule was fixed before looking: "below the line" only if the 95% range excludes zero.

In [4]:
show("src/o7_trend.py", "probit", "residual", "main")

# ---- src/o7_trend.py, lines 23-24
def probit(a):
    return norm.ppf(np.clip(a, 1e-4, 1 - 1e-4))

# ---- src/o7_trend.py, lines 35-40
def residual(X, Y, ref, target, idx, axis=probit, inverse=norm.cdf):
    x = np.array([axis(X[k][idx].mean()) for k in ref])
    y = np.array([axis(Y[k][idx].mean()) for k in ref])
    slope, icept = np.polyfit(x, y, 1)
    predicted = inverse(icept + slope * axis(X[target][idx].mean()))
    return 100 * (Y[target][idx].mean() - predicted)

# ---- src/o7_trend.py, lines 43-71
def main():
    rng = np.random.default_rng(0)
    out = {}
    for fam, conds in (("noise", NOISE), ("all 14", ALL14)):
        for s in (0, 1, 2):
            ref = SWEEP + [f"A-s{s}", f"C-s{s}"]
            keys = ref + [f"B-s{s}"]
            X = {k: clean(k) for k in keys}; Y = {k: damaged(k, conds) for k in keys}
            full = residual(X, Y, ref, f"B-s{s}", np.arange(N))
            boots = [residual(X, Y, ref, f"B-s{s}", rng.integers(0, N, N)) for _ in range(2000)]
   

In [5]:
import subprocess
print(subprocess.run([sys.executable, "src/o7_trend.py"], capture_output=True, text=True, check=True).stdout)

noise  seed 0: ternary -5.03 pts from the line, 95% range [-8.82, -1.81]  excludes 0
noise  seed 1: ternary -4.56 pts from the line, 95% range [-9.02, -0.61]  excludes 0
noise  seed 2: ternary -4.40 pts from the line, 95% range [-8.58, -1.14]  excludes 0
all 14 seed 0: ternary -2.77 pts from the line, 95% range [-5.04, -0.78]  excludes 0
all 14 seed 1: ternary -2.30 pts from the line, 95% range [-4.88, -0.13]  excludes 0
all 14 seed 2: ternary -2.84 pts from the line, 95% range [-4.95, -0.95]  excludes 0

Second check on noise: 98.3% range (Bonferroni, 3 seeds), probit axes and plain axes
seed 0: probit 98.3% [-9.47, -0.54]  |  plain axes -4.70, 98.3% [-9.26, +0.70]
seed 1: probit 98.3% [-9.68, +0.79]  |  plain axes -4.13, 98.3% [-9.65, +1.97]
seed 2: probit 98.3% [-9.27, +0.42]  |  plain axes -4.04, 98.3% [-9.15, +1.63]



**Reading it:** ternary sits 4–5 points below the line in every seed, and each 95% range excludes zero. But
with a stricter range for testing three seeds (Bonferroni, 98.3%) two of three seeds include zero, and noise is
a damage type where the trend often breaks down (Miller et al.). So this is reported as consistent, not proven.
The full check is in `results.md`.